In [ ]:
import os

from essential.gpu_utils import select_best_gpus

select_best_gpus(1)
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"


import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import os
import scanpy as sc
import plotnine as gg

import matplotlib.pyplot as plt
import plotly.express as px
import matplotlib.colors as mcolors
from tqdm import tqdm
import scipy.stats as st
from statsmodels.stats.multitest import multipletests

tab10_colors = plt.get_cmap("tab10").colors
tab10_hex = [mcolors.to_hex(c) for c in tab10_colors]
plt.rcParams["svg.fonttype"] = "none"

In [ ]:
adata = sc.read_h5ad(
    "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.scvi.h5ad"
)
adata.X = adata.layers["reads"].copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

adata.obs["transcript_UMAP1"] = adata.obsm["X_umap"][:, 0]
adata.obs["transcript_UMAP2"] = adata.obsm["X_umap"][:, 1]
adata.obs["target_"] = adata.obs["target"].astype(str)

In [ ]:
adata.obs

In [ ]:
adata_ctrl = adata[adata.obs["annotated_cluster"] == "control-like"].copy()

In [ ]:
from scvi.model import SCVI

SCVI.setup_anndata(adata_ctrl, layer="reads", batch_key="rt_bc")
model = SCVI(adata_ctrl)
model.train()

In [ ]:
z = model.get_latent_representation()
adata_ctrl.obsm["X_scvi_ctrl"] = z
sc.pp.neighbors(adata_ctrl, use_rep="X_scvi_ctrl")
sc.tl.umap(adata_ctrl)
sc.pl.umap(adata_ctrl)

In [ ]:
adata_ctrl.obs["transcript_ctrl_UMAP1"] = adata_ctrl.obsm["X_umap"][:, 0]
adata_ctrl.obs["transcript_ctrl_UMAP2"] = adata_ctrl.obsm["X_umap"][:, 1]

In [ ]:
(
    gg.ggplot(
        adata_ctrl.obs,
    )
    + gg.geom_point(
        gg.aes(x="transcript_ctrl_UMAP1", y="transcript_ctrl_UMAP2", color="rt_bc"),
        size=0.5,
    )
    + gg.theme_minimal()
    + gg.theme(figure_size=(12, 10))
)

In [ ]:
adata_sub = adata_ctrl.obs.loc[lambda x: x["target"].astype(str).str.startswith("rpo")].copy()
adata_sub["target"] = adata_sub["target"].astype(str)

(
    gg.ggplot(
        adata_ctrl.obs,
    )
    + gg.geom_point(
        gg.aes(x="transcript_ctrl_UMAP1", y="transcript_ctrl_UMAP2"),
        size=0.5,
    )
    + gg.geom_point(
        adata_sub,
        gg.aes(x="transcript_ctrl_UMAP1", y="transcript_ctrl_UMAP2", color="target"),
        size=3,
    )
    + gg.theme_minimal()
    + gg.theme(figure_size=(12, 10))
)

In [ ]:
adata_sub = adata_ctrl.obs.loc[
    lambda x: x["target"].astype(str).isin(["gyrA", "gyrB", "parC", "parE"])
].copy()
adata_sub["target"] = adata_sub["target"].astype(str)

(
    gg.ggplot(
        adata_ctrl.obs,
    )
    + gg.geom_point(
        gg.aes(x="transcript_ctrl_UMAP1", y="transcript_ctrl_UMAP2"),
        size=0.5,
    )
    + gg.geom_point(
        adata_sub,
        gg.aes(x="transcript_ctrl_UMAP1", y="transcript_ctrl_UMAP2", color="target"),
        size=3,
    )
    + gg.theme_minimal()
    + gg.theme(figure_size=(12, 10))
)